In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ==========================================
# 1. LOAD PROCESSED DATA
# ==========================================
print("Loading processed data...")
data = torch.load("data/processed/processed_data.pt")

src_train, evt_train, cat_train, gap_train, y_train = data["train"]
src_test, evt_test, cat_test, gap_test, y_test, test_ids = data["test"]
source_vocab, event_vocab, category_vocab = data["vocabs"]
truth_test = data["truth_test"]

# ==========================================
# 2. DATASET (stores variable-length tensors as-is)
# ==========================================
class SeqDataset(Dataset):
    def __init__(self, src, evt, cat, gaps, y=None):
        self.src, self.evt, self.cat, self.gaps = src, evt, cat, gaps
        self.y = y

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        if self.y is None:
            return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx]
        return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.y[idx]

# ==========================================
# 3. DYNAMIC PER-BATCH PADDING (pad only to the longest item IN the batch)
# ==========================================
def collate_fn(batch):
    has_labels = len(batch[0]) == 5
    if has_labels:
        src, evt, cat, gaps, y = zip(*batch)
    else:
        src, evt, cat, gaps = zip(*batch)

    lengths = torch.LongTensor([len(s) for s in src])

    src = pad_sequence(src, batch_first=True, padding_value=0)
    evt = pad_sequence(evt, batch_first=True, padding_value=0)
    cat = pad_sequence(cat, batch_first=True, padding_value=0)
    gaps = pad_sequence(gaps, batch_first=True, padding_value=0.0)

    if has_labels:
        y = torch.stack(y)
        return src, evt, cat, gaps, lengths, y
    return src, evt, cat, gaps, lengths

# ==========================================
# 4. LENGTH-BUCKETED BATCHING
# ==========================================
# Groups students of similar sequence length into the same batch so
# dynamic padding wastes as little compute as possible, then shuffles
# the ORDER of batches each epoch (not the students within a batch).
class BucketBatchSampler(Sampler):
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = np.argsort(self.lengths)
        batches = [indices[i:i+self.batch_size].tolist() for i in range(0, len(indices), self.batch_size)]
        if self.shuffle:
            np.random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

train_lengths = [len(s) for s in src_train]
test_lengths = [len(s) for s in src_test]

train_sampler = BucketBatchSampler(train_lengths, batch_size=128, shuffle=True)
test_sampler = BucketBatchSampler(test_lengths, batch_size=128, shuffle=False)

train_loader = DataLoader(SeqDataset(src_train, evt_train, cat_train, gap_train, y_train),
                           batch_sampler=train_sampler, collate_fn=collate_fn)
test_loader = DataLoader(SeqDataset(src_test, evt_test, cat_test, gap_test, y_test),
                          batch_sampler=test_sampler, collate_fn=collate_fn)

# ==========================================
# 5. MODEL (attention removed - classifier reads the BiLSTM's final hidden state)
# ==========================================
class BiLSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)

        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)
        # No attention layer. The classifier reads directly off the
        # concatenated final forward + backward hidden states (128-dim).
        self.fc = nn.Linear(128, 1)

    def forward(self, src, evt, cat, gaps, lengths):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)

        # h: (num_directions, batch, hidden) for a 1-layer BiLSTM
        h_cat = torch.cat([h[0], h[1]], dim=1)  # (batch, 128)

        return torch.sigmoid(self.fc(h_cat)).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMModel().to(device)

# ==========================================
# 6. TRAINING & EVALUATION
# ==========================================
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

print(f"Training on Device: {device}...")
for epoch in range(3):
    model.train()
    total_loss = 0
    for src, evt, cat, gaps, lengths, labels in train_loader:
        src, evt, cat, gaps, labels = [t.to(device) for t in (src, evt, cat, gaps, labels)]

        optimizer.zero_grad()
        outputs = model(src, evt, cat, gaps, lengths)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

print("\nEvaluating Test Data...")
model.eval()
preds = []
with torch.no_grad():
    for src, evt, cat, gaps, lengths, _ in test_loader:
        src, evt, cat, gaps = [t.to(device) for t in (src, evt, cat, gaps)]
        outputs = model(src, evt, cat, gaps, lengths)
        preds.extend(outputs.cpu().numpy())

# Predictions come out in bucketed (length-sorted) order, not the
# original student order - rebuild the matching id order the same way.
ordered_ids = []
for batch in BucketBatchSampler(test_lengths, batch_size=128, shuffle=False):
    ordered_ids.extend([test_ids[i] for i in batch])

eval_df = pd.DataFrame({"student_id": ordered_ids, "pred": preds}).merge(truth_test, on="student_id")
eval_df["predicted_dropout"] = (eval_df["pred"] > 0.5).astype(int)

print(f"Accuracy  : {accuracy_score(eval_df['dropout'], eval_df['predicted_dropout']):.4f}")
print(f"Precision : {precision_score(eval_df['dropout'], eval_df['predicted_dropout']):.4f}")
print(f"Recall    : {recall_score(eval_df['dropout'], eval_df['predicted_dropout']):.4f}")
print(f"F1 Score  : {f1_score(eval_df['dropout'], eval_df['predicted_dropout']):.4f}")
print(f"AUC       : {roc_auc_score(eval_df['dropout'], eval_df['pred']):.4f}")